# Modelo complejo

In [1]:
from importacion_preprocesado import download_and_load_data, preprocesamiento


tamany_img = (128,128) # tamaño reducido para colab, porque si no supera la ram
X, y = download_and_load_data(target_size=tamany_img)

X_train, X_val, X_test, y_train, y_val, y_test = preprocesamiento(X, y)

Dataset ya existe, solo se van a cargar las imágenes.
X shape: (4217, 128, 128, 3) y shape: (4217,)


In [2]:
# Importar la función
import sys
sys.path.append('../')

from models.complejo_cnn import create_model


# Crear el modelo
model = create_model(input_shape=(128,128,3), num_classes=4, l_rate=0.001)

c:\Users\roger\miniconda3\envs\dl\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
hist = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100
)

Epoch 1/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 76s 930ms/step - accuracy: 0.3887 - loss: 1.7664 - val_accuracy: 0.2630 - val_loss: 1.6230
Epoch 2/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 74s 920ms/step - accuracy: 0.4666 - loss: 1.5886 - val_accuracy: 0.2737 - val_loss: 1.6362
Epoch 3/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 74s 926ms/step - accuracy: 0.4725 - loss: 1.5393 - val_accuracy: 0.3578 - val_loss: 1.5760
Epoch 4/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 73s 914ms/step - accuracy: 0.4670 - loss: 1.5301 - val_accuracy: 0.4419 - val_loss: 1.5478
Epoch 5/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 71s 888ms/step - accuracy: 0.4947 - loss: 1.4714 - val_accuracy: 0.3709 - val_loss: 1.6121
Epoch 6/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 72s 905ms/step - accuracy: 0.4891 - loss: 1.4424 - val_accuracy: 0.4633 - val_loss: 1.5751
Epoch 7/100
13/80 ━━━━━━━━━━━━━━━━━━━━ 1:04 964ms/step - accuracy: 0.5370 - loss: 1.4591

In [ ]:
import matplotlib.pyplot as plt
# Visualizacion evolucion loss durante el entrenamiento
plt.plot(hist.history['loss'],label="loss")
plt.plot(hist.history['val_loss'],label="val_loss")
plt.legend()
plt.show()

# Visualizacion de accuracy durante el entrenamiento
plt.plot(hist.history["accuracy"], label= "loss")
plt.plot(hist.history["val_accuracy"], label ="val_accuracy")
plt.legend()
plt.show()

In [ ]:
model.summary()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

datasets = {
    "Train": (X_train, y_train),
    "Val": (X_val, y_val),
    "Test": (X_test, y_test)
}

def evaluate(model, X, y):
    #Convertir a enteros
    y_true = np.argmax(y, axis=1)

    y_pred_probs = model.predict(X)
    y_pred = np.argmax(y_pred_probs, axis=1)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    return acc, f1


def crear_df_metricas(model, datasets):
    results = {}
    for split_name, (X, y) in datasets.items():
        acc, f1 = evaluate(model, X, y)
        results[split_name] = [acc, f1]
    
    df = pd.DataFrame(results, index=["Accuracy", "F1"])
    return df

df = crear_df_metricas(model, datasets)
df.round(2)


In [ ]:
def plot_barra(metrica, titulo, color):

    plt.figure(figsize=(6,4))
    bars = plt.bar(metrica.index, metrica.values, color=color)
    plt.bar_label(bars)
    plt.title(f"{titulo}: Train vs Validation vs Test")
    plt.ylabel(titulo)
    plt.ylim(0,1)
    plt.grid(axis='y', linestyle='--')
    plt.show()


def plots_metricas(df):
    #Extraemos la fila
    accuracy = df.loc["Accuracy"]
    f1 = df.loc["F1"]

    #Grafico de Accuracy
    plot_barra(accuracy, titulo = "Accuracy", color="blue")

    #Grafico de 
    plot_barra(f1, titulo ="F1", color = "red")

In [ ]:
plots_metricas(df)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(model.predict(X_test), axis=1)

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm).plot(cmap="Blues")